In [ ]:
"""
Notebook de Resultados - Práctica Ajedrez
===================================
Este notebook carga, agrega y visualiza los resultados de los modelos entrenados
(KNN, SVM, Naive Bayes, Random Forest) y sus combinaciones mediante ensembles
(Votación, Media, Mediana) sobre diferentes versiones del dataset.

Genera tablas resumen y gráficas comparativas para el análisis del rendimiento.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import os
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, f1_score, recall_score, precision_score
import warnings

# Ignoramos warnings de división por cero (común en validación cruzada si una clase no aparece)
warnings.filterwarnings('ignore')

## Configuración

En esta sección definimos las constantes principales del análisis:
- **METHODS**: Lista de algoritmos evaluados (modelos base + ensembles)
- **DATASETS**: Variantes del conjunto de datos (original, normalizado, estandarizado, con y sin PCA)
- **METRICS**: Métricas de evaluación calculadas para cada modelo
- **Rutas**: Ubicación de los archivos de predicciones y métricas generados por `eval.ipynb`

In [ ]:
# Algoritmos y Ensembles
METHODS = ["KNN", "SVM", "NaiveBayes", "RandomForest", "Ensemble_Votacion", "Ensemble_Media", "Ensemble_Mediana"]

# Variantes de Datasets solicitadas 
DATASETS = [
    "original", "estandarizado", "normalizado",
    "original_PCA95", "original_PCA80",
    "estandarizado_PCA95", "estandarizado_PCA80",
    "normalizado_PCA95", "normalizado_PCA80"
]

# Métricas requeridas 
METRICS = ["Exactitud", "F1", "Sensibilidad", "Especificidad", "Precision", "FNR", "FPR", "AUC"]

FOLDS = 5
PATH_METRICAS = "./metricas"
PATH_PREDICCIONES = "./predicciones"

## Carga y Agregación de Datos

Esta sección contiene las funciones para:
1. **Calcular métricas personalizadas**: A partir de las predicciones y probabilidades, calcula F1, Sensibilidad, Exactitud, Especificidad, Recall, Precision, FNR, FPR y AUC.
2. **Cargar y agregar resultados**: Lee los archivos de métricas generados por `eval.ipynb` para cada método, dataset y fold, calculando la media y desviación estándar de cada métrica.

In [ ]:
def calculate_custom_metrics_lichess(y_true, y_pred, y_proba):
    # Clases específicas del proyecto de ajedrez
    CLASES = ['1-0', '0-1', '1/2-1/2']
    
    acc = accuracy_score(y_true, y_pred) # Exactitud
    f1 = f1_score(y_true, y_pred, average='macro') # F1-score
    rec = recall_score(y_true, y_pred, average='macro') # Sensibilidad / Recall
    prec = precision_score(y_true, y_pred, average='macro') # Precisión
    
    cm = confusion_matrix(y_true, y_pred, labels=CLASES)
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    epsilon = 1e-7 
    spec = np.mean(TN / (FP + TN + epsilon)) # Especificidad
    fnr = np.mean(FN / (TP + FN + epsilon))  # Tasa Falsos Negativos
    fpr = np.mean(FP / (FP + TN + epsilon))  # Tasa Falsos Positivos
    
    auc_val = 0
    if y_proba is not None:
        try:
            auc_val = roc_auc_score(y_true, y_proba, multi_class='ovr', labels=CLASES) # AUC
        except: pass

    return {
        "Exactitud": acc, "F1": f1, "Sensibilidad": rec, "Recall": rec,
        "Precision": prec, "Especificidad": spec, "FNR": fnr, "FPR": fpr, "AUC": auc_val
    }

def load_and_aggregate_results():
    aggregated_data = []
    for method in METHODS:
        for dataset in DATASETS:
            fold_metrics = {m: [] for m in METRICS}
            for k in range(1, FOLDS + 1):
                # Intentamos cargar métricas directas
                filename = f"{PATH_METRICAS}/{method}/{dataset}/{method}{k}_metrics.csv"
                try:
                    df = pd.read_csv(filename)
                    for m in METRICS:
                        if m in df.columns: fold_metrics[m].append(df[m].values[0])
                except FileNotFoundError:
                    # Si no, cargamos predicciones y calculamos
                    pred_file = f"{PATH_PREDICCIONES}/{method}/{dataset}/{method}{k}_predicts.csv"
                    if os.path.exists(pred_file):
                        df_p = pd.read_csv(pred_file)
                        prob_cols = [c for c in df_p.columns if 'prob_' in c]
                        m_dict = calculate_custom_metrics_lichess(df_p['y_true'], df_p['y_pred'], df_p[prob_cols].values if prob_cols else None)
                        for m in METRICS: fold_metrics[m].append(m_dict[m])

            if len(fold_metrics["F1"]) > 0:
                row = {'Method': method, 'Dataset': dataset}
                for m in METRICS:
                    mean_val, std_val = np.mean(fold_metrics[m]), np.std(fold_metrics[m])
                    row[f"{m}_mean"], row[f"{m}_std"] = mean_val, std_val
                    row[f"{m}_display"] = f"{mean_val:.4f} $\\pm$ {std_val:.4f}"
                aggregated_data.append(row)
    return pd.DataFrame(aggregated_data)

df_results = load_and_aggregate_results()

## Generación de Tablas para LaTeX

Esta sección genera tablas formateadas para incluir directamente en documentos LaTeX:

1. **Tablas por método**: Para cada algoritmo (KNN, SVM, etc.), genera una tabla donde las filas son los datasets y las columnas son las métricas. Muestra media ± desviación estándar.

2. **Tabla resumen F1-Score**: Tabla comparativa global donde las filas son los métodos y las columnas son los datasets, mostrando únicamente el F1-Score para facilitar la comparación rápida entre algoritmos.

In [ ]:
def generate_latex_tables(df):
    for method in METHODS:
        subset = df[df['Method'] == method].copy()
        if subset.empty: continue
        
        cols_display = [f"{m}_display" for m in METRICS]
        table = subset.set_index('Dataset')[cols_display]
        table.columns = METRICS
        
        print(f"\n% --- TABLA LATEX: {method} ---")
        print(table.to_latex(escape=False, caption=f'Rendimiento de {method}', label=f'tab:{method.lower()}'))

generate_latex_tables(df_results)

## Generación de Gráficas Comparativas

Esta sección genera visualizaciones para analizar el rendimiento de los modelos:

### Gráficas generadas:

1. **FPR vs FNR** (Tasa de Falsos Positivos vs Tasa de Falsos Negativos):
   - Permite visualizar el trade-off entre errores tipo I y tipo II
   - Idealmente, ambas tasas deben ser bajas (cercanas a 0)

2. **Precision vs Recall**:
   - Muestra el equilibrio entre precisión y cobertura
   - Importante para problemas con clases desbalanceadas

3. **Accuracy vs F1-Score**:
   - Compara la exactitud global con el F1 ponderado
   - F1 es más robusto ante clases desbalanceadas

Cada punto representa una combinación método-dataset, coloreado por algoritmo.

In [ ]:
def plot_comparisons_lichess(df):
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Marcadores para diferenciar base de ensembles
    markers = {'KNN': 'o', 'SVM': 's', 'NaiveBayes': '^', 'RandomForest': 'D',
               'Ensemble_Votacion': '*', 'Ensemble_Media': 'P', 'Ensemble_Mediana': 'X'}

    plot_pairs = [
        ('FPR_mean', 'FNR_mean', 'FPR vs FNR (Tasas de Error)'),
        ('Sensibilidad_mean', 'Precision_mean', 'Precision vs Sensibilidad'),
        ('F1_mean', 'Exactitud_mean', 'Exactitud vs F1-Score')
    ]
    
    for ax, (x_m, y_m, title) in zip(axes, plot_pairs):
        for method in METHODS:
            sub = df[df['Method'] == method]
            if not sub.empty:
                ax.scatter(sub[x_m], sub[y_m], label=method, marker=markers.get(method, 'o'), s=120, alpha=0.8, edgecolors='k')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel(x_m.replace('_mean', ''))
        ax.set_ylabel(y_m.replace('_mean', ''))

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('comparativa_final_lichess.png', dpi=150)
    plt.show()

plot_comparisons_lichess(df_results)

In [ ]:
def plot_f1_analysis(df):
    """
    Genera un Heatmap y un gráfico de barras comparativo enfocados en el F1-Score.
    """
    plt.style.use('seaborn-v0_8-whitegrid')
    
    # Pivotamos el dataframe para tener Métodos en filas y Datasets en columnas
    df_pivot = df.pivot(index='Method', columns='Dataset', values='F1_mean')
    
    # Reordenar columnas para que sigan el orden lógico de DATASETS definido al inicio
    df_pivot = df_pivot.reindex(columns=DATASETS, index=METHODS)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14))

    sns.heatmap(df_pivot, annot=True, fmt=".4f", cmap="YlGnBu", ax=ax1, cbar_kws={'label': 'F1-Score Mean'})
    ax1.set_title('Heatmap: Rendimiento F1-Score por Método y Dataset', fontsize=15, fontweight='bold', pad=20)
    ax1.set_xlabel('Variante del Dataset', fontsize=12)
    ax1.set_ylabel('Algoritmo / Ensemble', fontsize=12)

    sns.barplot(data=df, x='Dataset', y='F1_mean', hue='Method', ax=ax2, palette='viridis')
    ax2.set_title('Comparación Global de F1-Score en todos los Datasets', fontsize=15, fontweight='bold', pad=20)
    ax2.set_xlabel('Variante del Dataset', fontsize=12)
    ax2.set_ylabel('F1-Score (Media)', fontsize=12)
    ax2.set_ylim(0, 1.1) # El F1-score suele estar entre 0 y 1
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Métodos')
    
    # Ajustar etiquetas del eje X para que no se solapen
    plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    plt.savefig('analisis_f1_detallado.png', dpi=150, bbox_inches='tight')
    plt.show()

# Ejecutar la nueva visualización
plot_f1_analysis(df_results)